In [ ]:
#certainty will be noise while training
df = roman_chord_analysis_df.drop("certainty", axis = 1)
#stepsto the end is an effective way for the model to understand time
df["steps_to_end"] = df.groupby("chorale_name").cumcount(ascending=False)
chords = order
chord_map = {j: i for i, j in enumerate(chords)}
quality = sorted(df.chord_quality.unique())
quality_map = {j: i for i, j in enumerate(quality)}
#inversions, already contains only the integers 0-5, so it does
#not need mapping

#grouping the dataset into lists
cgroup = df.groupby("chorale_name").agg(list)

#roman chords numbers
roman = cgroup["roman_scale_degree"].tolist()
#chord quality
qual = cgroup["chord_quality"].tolist()
#inversion
inversion = cgroup["inversion"].tolist()
#beat_stength
beat = cgroup["beat_strength"].tolist()
#has7
has7 = cgroup["has7"].tolist()
#sense of time - steps until the end of the chorale
steps = cgroup["steps_to_end"].tolist()

In [ ]:
#LSTM Model
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score
import numpy as np

#metrics equation
def full_metrics(model, X_test, y_test, n_classes=14):
    probs = model.predict(X_test, verbose=0)
    pred  = probs.argmax(axis=1)

    ce   = -np.mean(np.log(probs[np.arange(len(y_test)), y_test] + 1e-12))
    top1 = (pred == y_test).mean()
    top3 = np.mean([y in row for y, row in
                    zip(y_test, np.argsort(-probs, axis=1)[:, :3])])
    maj  = np.bincount(y_test).max() / len(y_test)
    f1   = f1_score(y_test, pred, average="macro")

    return {"CE": ce, "PP": np.exp(ce), "top1": top1, "top3": top3,
            "macroF1": f1, "majority": maj, "random_CE": np.log(n_classes)}

#make a list for each predictor/target variable
roman = cgroup["roman_scale_degree"].tolist()
qual = cgroup["chord_quality"].tolist()
inversion = cgroup["inversion"].tolist()
beat = cgroup["beat_strength"].tolist()
has7 = cgroup["has7"].tolist()
steps = cgroup["steps_to_end"].tolist()

#spliting
rng = np.random.default_rng(25267220)
location = rng.permutation(len(cgroup))

#we split the data into test and train and train
#with consistency around 14%, 14%, and, 72% accordingly
test_slice  = slice(0, 50)
val_slice   = slice(50, 100)
train_slice = slice(100, None)

#spliting using location
roman = [roman[i] for i in location]
qual = [qual[i] for i in location]
inversion = [inversion[i] for i in location]
beat = [beat[i] for i in location]
has7 = [has7[i] for i in location]
steps = [steps[i] for i in location]

#This list will save results
results = []

#window size
for wsize in [4, 8, 12]:
 for units in [32, 64, 128]:
  for dropout in [0.0, 0.3]:

    #Transforming the data (predictors and target variables) into a suited form
    #for the model to process.
    def make_windows(dslice, wsize):
      #for each predictor variable
      #we will turn the variable into a list (grouped by chorale name)
      #that each row contains
      #the list of the values of that variable for each chorale,
      #since we predict each time, the continuation of a chorale.
      Xr, Xq, Xi, Xh, Xb, Xs, y = [], [], [], [], [], [], []
      for r, q, i, b, h, s in zip(roman[dslice], qual[dslice],
                                  inversion[dslice],
                                  beat[dslice], has7[dslice], steps[dslice]):
        #we turn roman degrees and quality into integers
        roman_map = [chord_map[j] for j in r]
        qual_map = [quality_map[j] for j in q]
        for j in range(len(roman_map) - wsize):
          Xr.append(roman_map[j:j+wsize])
          Xq.append(qual_map[j:j+wsize])
          Xi.append(i[j:j+wsize])
          Xb.append(b[j:j+wsize])
          Xh.append(h[j:j+wsize])
          Xs.append(s[j:j+wsize])
          y.append(roman_map[j+wsize])
      return([np.array(Xr), np.array(Xq), np.array(Xi), np.array(Xh),
              np.array(Xb).reshape(-1, wsize, 1),
              np.array(Xs).reshape(-1, wsize, 1)], np.array(y))

    #the train, val sets
    X_train, y_train = make_windows(train_slice, wsize)
    X_val, y_val = make_windows(val_slice, wsize)
    X_test, y_test = make_windows(test_slice, wsize)

    #Now we have to get the data ready for the layers
    Xr = layers.Input(shape = (wsize,))
    Xq = layers.Input(shape = (wsize,))
    Xi = layers.Input(shape = (wsize,))
    Xh = layers.Input(shape = (wsize,))
    Xb = layers.Input(shape = (wsize,1))
    Xs = layers.Input(shape = (wsize,1))#continuous

    #Now we have to decide the dimensionality for each chorale
    embroman = layers.Embedding(14, 16)(Xr)
    embqual = layers.Embedding(5, 6)(Xq)
    embinv = layers.Embedding(6, 8)(Xi)
    embh7 = layers.Embedding(2, 4)(Xh)

    #layers for each variable
    x = layers.concatenate([embroman, embqual, embinv, embh7, Xb, Xs])
    #LSTM model
    x = layers.LSTM(units, dropout = dropout)(x)
    #output layer
    out = layers.Dense(14, activation = "softmax")(x)

    #specifying the model
    model = keras.Model([Xr, Xq, Xi, Xh, Xb, Xs], out)
    #optimizer
    optimizer = keras.optimizers.Adam(learning_rate = 0.001)
    #early stopping
    early_stopping = keras.callbacks.EarlyStopping(patience = 10,
                                                   restore_best_weights=True)
    #compiling the model
    model.compile(
        optimizer = optimizer, loss = "sparse_categorical_crossentropy",
        metrics = ["accuracy"])
    #fitting the model to the variables
    history = model.fit(X_train, y_train, validation_data = (X_val, y_val),
                        epochs = 50, batch_size = 128,
                        callbacks = [early_stopping], verbose = 0)

    #evaluating the final loss and accuracy
    loss, accuracy = model.evaluate(X_val, y_val, verbose = 0)

    #printing the final results accuracy loss - hyperparameters
    print("Number of units:", units, "dropout:", dropout, "\n\n",
          "window size:", wsize, "accuracy:", accuracy, "loss:", loss)

    #creating the result list
    scores = full_metrics(model, X_val, y_val)
    scores.update({"wsize": wsize, "units": units, "dropout": dropout})
    results.append(scores)

    #printing the final results for each combination of hyperparameters
    print(f"w{wsize} u{units} d{dropout} → CE {scores['CE']:.3f} | "
          f"PP {scores['PP']:.2f} | top1 {scores['top1']:.3f} | "
          f"top3 {scores['top3']:.3f} | F1 {scores['macroF1']:.3f}")

In [ ]:
#GRU Model
#make a list for each predictor/target variable
roman = cgroup["roman_scale_degree"].tolist()
qual = cgroup["chord_quality"].tolist()
inversion = cgroup["inversion"].tolist()
beat = cgroup["beat_strength"].tolist()
has7 = cgroup["has7"].tolist()
steps = cgroup["steps_to_end"].tolist()

#spliting
rng = np.random.default_rng(25267220)
location = rng.permutation(len(cgroup))

#we split the data into test and train and train
#with consistency around 14%, 14%, and, 72% accordingly
test_slice  = slice(0, 50)
val_slice   = slice(50, 100)
train_slice = slice(100, None)

#spliting using location
roman = [roman[i] for i in location]
qual = [qual[i] for i in location]
inversion = [inversion[i] for i in location]
beat = [beat[i] for i in location]
has7 = [has7[i] for i in location]
steps = [steps[i] for i in location]

#window size
for wsize in [4, 8, 12]:
 for units in [32, 64, 128]:
  for dropout in [0.0, 0.3]:

    #Transforming the data (predictors and target variables) into a suited form
    #for the model to process.
    def make_windows(dslice, wsize):
      #for each predictor variable
      #we will turn the variable into a list (grouped by chorale name)
      #that each row contains
      #the list of the values of that variable for each chorale,
      #since we predict each time, the continuation of a chorale.
      Xr, Xq, Xi, Xh, Xb, Xs, y = [], [], [], [], [], [], []
      for r, q, i, b, h, s in zip(roman[dslice], qual[dslice],
                                  inversion[dslice],
                                  beat[dslice], has7[dslice], steps[dslice]):
        #we turn roman degrees and quality into integers
        roman_map = [chord_map[j] for j in r]
        qual_map = [quality_map[j] for j in q]
        for j in range(len(roman_map) - wsize):
          Xr.append(roman_map[j:j+wsize])
          Xq.append(qual_map[j:j+wsize])
          Xi.append(i[j:j+wsize])
          Xb.append(b[j:j+wsize])
          Xh.append(h[j:j+wsize])
          Xs.append(s[j:j+wsize])
          y.append(roman_map[j+wsize])
      return([np.array(Xr), np.array(Xq), np.array(Xi), np.array(Xh),
              np.array(Xb).reshape(-1, wsize, 1),
              np.array(Xs).reshape(-1, wsize, 1)], np.array(y))

    #the train, val sets
    X_train, y_train = make_windows(train_slice, wsize)
    X_val, y_val = make_windows(val_slice, wsize)
    X_test, y_test = make_windows(test_slice, wsize)

    #Now we have to get the data ready for the layers
    Xr = layers.Input(shape = (wsize,))
    Xq = layers.Input(shape = (wsize,))
    Xi = layers.Input(shape = (wsize,))
    Xh = layers.Input(shape = (wsize,))
    Xb = layers.Input(shape = (wsize,1))
    Xs = layers.Input(shape = (wsize,1))#continuous

    #Now we have to decide the dimensionality for each chorale
    embroman = layers.Embedding(14, 16)(Xr)
    embqual = layers.Embedding(5, 6)(Xq)
    embinv = layers.Embedding(6, 8)(Xi)
    embh7 = layers.Embedding(2, 4)(Xh)

    x = layers.concatenate([embroman, embqual, embinv, embh7, Xb, Xs])

    x = layers.GRU(units, dropout = dropout)(x)
    #output layer
    out = layers.Dense(14, activation = "softmax")(x)

    #specifying the model
    model = keras.Model([Xr, Xq, Xi, Xh, Xb, Xs], out)
    #optimizer
    optimizer = keras.optimizers.Adam(learning_rate = 0.001)
    #early stopping
    early_stopping = keras.callbacks.EarlyStopping(patience = 10,
                                                   restore_best_weights=True)
    #compiling the model
    model.compile(
        optimizer = optimizer, loss = "sparse_categorical_crossentropy",
        metrics = ["accuracy"])
    #fitting the model to the variables
    history = model.fit(X_train, y_train, validation_data = (X_val, y_val),
                        epochs = 50, batch_size = 128,
                        callbacks = [early_stopping], verbose = 0)

    #evaluating the final loss and accuracy
    loss, accuracy = model.evaluate(X_val, y_val, verbose = 0)

    #printing the final results accuracy loss - hyperparameters
    print("Number of units:", units, "dropout:", dropout, "\n\n",
          "window size:", wsize, "accuracy:", accuracy, "loss:", loss)

    #creating the result list
    scores = full_metrics(model, X_val, y_val)
    scores.update({"wsize": wsize, "units": units, "dropout": dropout})
    results.append(scores)

    #printing the final results for each combination of hyperparameters
    print(f"w{wsize} u{units} d{dropout} → CE {scores['CE']:.3f} | "
          f"PP {scores['PP']:.2f} | top1 {scores['top1']:.3f} | "
          f"top3 {scores['top3']:.3f} | F1 {scores['macroF1']:.3f}")

In [ ]:
#Linear Model

#make a list for each predictor/target variable
roman = cgroup["roman_scale_degree"].tolist()
qual = cgroup["chord_quality"].tolist()
inversion = cgroup["inversion"].tolist()
beat = cgroup["beat_strength"].tolist()
has7 = cgroup["has7"].tolist()
steps = cgroup["steps_to_end"].tolist()

#spliting
rng = np.random.default_rng(25267220)
location = rng.permutation(len(cgroup))

#we split the data into test and train and train
#with consistency around 14%, 14%, and, 72% accordingly
test_slice  = slice(0, 50)
val_slice   = slice(50, 100)
train_slice = slice(100, None)

#spliting using location
roman = [roman[i] for i in location]
qual = [qual[i] for i in location]
inversion = [inversion[i] for i in location]
beat = [beat[i] for i in location]
has7 = [has7[i] for i in location]
steps = [steps[i] for i in location]


for wsize in [4, 8, 12]:
 for units in [32, 64, 128]:
  for dropout in [0.0, 0.3]:

    #Transforming the data (predictors and target variables) into a suited form
    #for the model to process.
    def make_windows(dslice, wsize):
      #for each predictor variable
      #we will turn the variable into a list (grouped by chorale name)
      #that each row contains
      #the list of the values of that variable for each chorale,
      #since we predict each time, the continuation of a chorale.
      Xr, Xq, Xi, Xh, Xb, Xs, y = [], [], [], [], [], [], []
      for r, q, i, b, h, s in zip(roman[dslice], qual[dslice],
                                  inversion[dslice],
                                  beat[dslice], has7[dslice], steps[dslice]):
        #we turn roman degrees and quality into integers
        roman_map = [chord_map[j] for j in r]
        qual_map = [quality_map[j] for j in q]
        for j in range(len(roman_map) - wsize):
          Xr.append(roman_map[j:j+wsize])
          Xq.append(qual_map[j:j+wsize])
          Xi.append(i[j:j+wsize])
          Xb.append(b[j:j+wsize])
          Xh.append(h[j:j+wsize])
          Xs.append(s[j:j+wsize])
          y.append(roman_map[j+wsize])
      return([np.array(Xr), np.array(Xq), np.array(Xi), np.array(Xh),
              np.array(Xb).reshape(-1, wsize, 1),
              np.array(Xs).reshape(-1, wsize, 1)], np.array(y))

    #the train, val sets
    X_train, y_train = make_windows(train_slice, wsize)
    X_val, y_val = make_windows(val_slice, wsize)
    X_test, y_test = make_windows(test_slice, wsize)

    #Now we have to get the data ready for the layers
    Xr = layers.Input(shape = (wsize,))
    Xq = layers.Input(shape = (wsize,))
    Xi = layers.Input(shape = (wsize,))
    Xh = layers.Input(shape = (wsize,))
    Xb = layers.Input(shape = (wsize,1))
    Xs = layers.Input(shape = (wsize,1))#continuous

    #Now we have to decide the dimensionality for each chorale
    embroman = layers.Embedding(14, 16)(Xr)
    embqual = layers.Embedding(5, 6)(Xq)
    embinv = layers.Embedding(6, 8)(Xi)
    embh7 = layers.Embedding(2, 4)(Xh)

    x = layers.concatenate([embroman, embqual, embinv, embh7, Xb, Xs])
    x = layers.Flatten()(x)
    #output layer
    out = layers.Dense(14, activation = "softmax")(x)

    #specifying the model
    model = keras.Model([Xr, Xq, Xi, Xh, Xb, Xs], out)
    #optimizer
    optimizer = keras.optimizers.Adam(learning_rate = 0.001)
    #early stopping
    early_stopping = keras.callbacks.EarlyStopping(patience = 10,
                                                    restore_best_weights=True)
    #compiling the model
    model.compile(
         optimizer = optimizer, loss = "sparse_categorical_crossentropy",
         metrics = ["accuracy"])
    #fitting the model to the variables
    history = model.fit(X_train, y_train, validation_data = (X_val, y_val),
                         epochs = 50, batch_size = 128,
                         callbacks = [early_stopping], verbose = 0)

    #evaluating the final loss and accuracy
    loss, accuracy = model.evaluate(X_val, y_val, verbose = 0)

    #printing the final results accuracy loss - hyperparameters
    print("Number of units:", units, "dropout:", dropout, "\n\n",
           "window size:", wsize, "accuracy:", accuracy, "loss:", loss)

    #creating the result list
    scores = full_metrics(model, X_val, y_val)
    scores.update({"wsize": wsize, "units": units, "dropout": dropout})
    results.append(scores)

    #printing the final results for each combination of hyperparameters
    print(f"w{wsize} u{units} d{dropout} → CE {scores['CE']:.3f} | "
           f"PP {scores['PP']:.2f} | top1 {scores['top1']:.3f} | "
           f"top3 {scores['top3']:.3f} | F1 {scores['macroF1']:.3f}")

In [ ]:
lengths = [len(r) for r in roman]
print("min:", min(lengths), "| median:", int(np.median(lengths)), "| max:", max(lengths))
print("  40:", sum(1 for l in lengths if l < 40), "", len(lengths))
print("  L=8 :", sum(max(0, l-8) for l in lengths))
print("  L=40:", sum(max(0, l-40) for l in lengths))